# 10 — Real-World Examples

This notebook ties everything together. We'll run the same set of queries through different strategies and compare the results.

> **Note:** Full execution requires a running backend with an LLM (OpenAI / Anthropic) and a populated vector store. If those are not available, we'll simulate realistic outputs.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.router.query_classifier import QueryClassifier
    from backend.adaptive_rag.router.strategy_selector import StrategySelector
    from backend.adaptive_rag.router.confidence_scorer import ConfidenceScorer
    print("✓ Imported router components")
except ImportError as e:
    print(f"⚠ Router import: {e}")

## 2. Query Suite

We'll test 6 representative queries:

In [ ]:
queries = [
    {
        "id": 1,
        "query": "What is the capital of France?",
        "description": "Simple factual — should use document RAG"
    },
    {
        "id": 2,
        "query": "Latest developments in quantum computing",
        "description": "Time-sensitive — should use web search"
    },
    {
        "id": 3,
        "query": "Compare Python and Rust for systems programming with recent community trends",
        "description": "Complex comparative + time-sensitive — hybrid"
    },
    {
        "id": 4,
        "query": "How do I deploy a machine learning model to production?",
        "description": "Procedural — document RAG with technical docs"
    },
    {
        "id": 5,
        "query": "What is the best programming language for beginners?",
        "description": "Opinion — direct LLM (no retrieval needed)"
    },
    {
        "id": 6,
        "query": "Explain the relationship between interest rates and inflation using economic data",
        "description": "Exploratory + graph signal — graph or hybrid"
    },
]

for q in queries:
    print(f"[{q['id']}] {q['query']}")
    print(f"    {q['description']}")
    print()

## 3. Classify All Queries

In [ ]:
classifier = QueryClassifier()
selector = StrategySelector()

print(f"{'ID':<4} {'Query':<40} {'Type':<14} {'Complexity':<12} {'Strategy':<18}")
print("─" * 88)
for q in queries:
    c = classifier.classify(q["query"])
    s = selector.select(c)
    print(f"{q['id']:<4} {q['query'][:39]:<40} {c.query_type:<14} {c.complexity:<12} {s:<18}")

## 4. Cross-Strategy Comparison (Simulated)

Let's simulate what each strategy would produce for the same query:

In [ ]:
import json

def simulate_strategy(name, query):
    base_answers = {
        "direct_llm": f"[Direct LLM] Answering '{query[:40]}' from internal knowledge...",
        "document_rag": f"[Document RAG] Retrieved 3 docs, generated answer for '{query[:40]}'",
        "web_search_rag": f"[Web RAG] Fetched 3 web results, generated answer for '{query[:40]}'",
        "hybrid_rag": f"[Hybrid RAG] Combined 2 docs + 2 web results for '{query[:40]}'",
        "self_rag": f"[Self-RAG] Iterated 2 times, generated answer for '{query[:40]}'",
        "corrective_rag": f"[Corrective RAG] Graded docs low, corrected via web, answered '{query[:40]}'",
    }
    return {
        "strategy": name,
        "query": query,
        "answer_snippet": base_answers.get(name, f"[{name}] processed query"),
        "confidence": ConfidenceScorer().score({"strategy": name, "sources": [{"content": "test"}] * 3, "answer": "test"}),
    }

query_id = 3
q = queries[query_id - 1]["query"]

print(f"Query: {q}\n")
for strat in ["direct_llm", "document_rag", "web_search_rag", "hybrid_rag", "self_rag", "corrective_rag"]:
    r = simulate_strategy(strat, q)
    print(f"  {r['strategy']:<18} confidence: {r['confidence']:.2f} | {r['answer_snippet'][:70]}...")
    print()

## 5. Multi-Query Comparison Table

Let's see which strategy is *selected* for each query and compare confidence scores across all strategies:

In [ ]:
strategies = ["direct_llm", "document_rag", "web_search_rag", "hybrid_rag", "self_rag", "corrective_rag"]

header = f"{'Query':<50} {'Selected':<18}"
for s in strategies:
    header += f" {s[:6]:<8}"
print(header)
print("─" * (68 + 8 * len(strategies)))

for q in queries:
    c = classifier.classify(q["query"])
    selected = selector.select(c)
    row = f"{q['query'][:49]:<50} {selected:<18}"
    for s in strategies:
        r = simulate_strategy(s, q["query"])
        row += f" {r['confidence']:.2f}    "
    print(row)

## 6. End-to-End: Simulated Pipeline Run

This demonstrates the full `AdaptiveWorkflow.run()` call path:

In [ ]:
async def simulated_pipeline_run(query):
    from backend.adaptive_rag.router.confidence_scorer import ConfidenceScorer
    
    # Step 1: Classify
    c = classifier.classify(query)
    
    # Step 2: Select strategy
    strategy = selector.select(c)
    
    # Step 3: Execute (simulated)
    result = simulate_strategy(strategy, query)
    
    # Step 4: Score confidence
    scorer = ConfidenceScorer()
    confidence = scorer.score({
        "strategy": strategy,
        "sources": [{"content": "sample"}],
        "answer": result["answer_snippet"],
        "documents": [{"source": "test"}] * 3,
    })
    
    return {
        "query": query,
        "classification": c.model_dump(),
        "selected_strategy": strategy,
        "answer": result["answer_snippet"],
        "confidence": confidence,
        "confidence_level": scorer.get_confidence_level(confidence),
    }

import asyncio

for q in queries[:3]:
    r = asyncio.run(simulated_pipeline_run(q["query"]))
    print(f"Query       : {r['query']}")
    print(f"Classified  : {r['classification']['query_type']} / {r['classification']['complexity']}")
    print(f"Strategy    : {r['selected_strategy']}")
    print(f"Confidence  : {r['confidence']:.2f} ({r['confidence_level']})")
    print(f"Answer      : {r['answer'][:100]}...")
    print()

## 7. Performance Analysis

### Costs & Latency Comparison

In [ ]:
performance = {
    "direct_llm":     {"latency": "low",    "cost": "low",    "freshness": "stale", "accuracy": "medium"},
    "document_rag":   {"latency": "medium", "cost": "medium", "freshness": "stale", "accuracy": "high"},
    "web_search_rag": {"latency": "high",   "cost": "medium", "freshness": "fresh", "accuracy": "medium"},
    "hybrid_rag":     {"latency": "high",   "cost": "high",   "freshness": "mixed", "accuracy": "high"},
    "self_rag":       {"latency": "high",   "cost": "high",   "freshness": "stale", "accuracy": "very high"},
    "corrective_rag": {"latency": "high",   "cost": "high",   "freshness": "mixed", "accuracy": "very high"},
}

print(f"{'Strategy':<20} {'Latency':<10} {'Cost':<10} {'Freshness':<10} {'Accuracy':<12}")
print("─" * 62)
for name, metrics in performance.items():
    print(f"{name:<20} {metrics['latency']:<10} {metrics['cost']:<10} {metrics['freshness']:<10} {metrics['accuracy']:<12}")

## 8. When to Use What

| Scenario | Recommended Strategy | Why |
|----------|--------------------|-----|
| "Define X", simple fact | `direct_llm` | Fastest, cheapest |
| "Explain Y" with internal docs | `document_rag` | High accuracy on known data |
| "Latest news on Z" | `web_search_rag` | Freshness required |
| Complex analysis with recent data | `hybrid_rag` | Best of both worlds |
| High-stakes accuracy needed | `self_rag` | Self-reflection reduces errors |
| Domain where docs may be incomplete | `corrective_rag` | Graceful web fallback |
| Entity relationships | `graph_rag` | Structured knowledge queries |

## 9. Production Considerations

1. **Caching** — Cache classification results and frequent queries
2. **Streaming** — Stream answers token‑by‑token in production
3. **Observability** — Log every node transition in the LangGraph
4. **A/B Testing** — Compare strategy performance on held‑out queries
5. **Fallback Chains** — If hybrid fails, fall back to document, then direct LLM
6. **Rate Limiting** — LLM calls and web search API quotas

## 10. Explore Further

| Resource | Link |
|----------|------|
| Backend source | `./backend/adaptive_rag/` |
| API endpoints | `./backend/api/` |
| LangGraph docs | https://langchain-ai.github.io/langgraph/ |
| Project README | `./README.md` |

---

**You've completed the Adaptive RAG pipeline walkthrough!** 🎉

You should now understand:
- How `QueryClassifier` profiles queries
- How `StrategySelector` picks the best approach
- How each strategy (document, web, hybrid, self, corrective) works
- How LangGraph orchestrates the full adaptive workflow